# Assignment 3: Comparative machine translation

This notebook is the persistent Google Colab entry point. **Input required:** select a GPU runtime and provide a Hugging Face token in Colab Secrets. It runs the repository modules rather than duplicating their implementations.

In [ ]:
# INPUT CELL: set REPO_REF only if you need a branch/commit other than main.
REPO_URL = 'https://github.com/sruthiidv/machine-translation-model-comparison.git'
REPO_REF = 'main'
from pathlib import Path
import os, subprocess
repo = Path('/content/machine-translation-model-comparison')
if not repo.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(repo)], check=True)
os.chdir(repo)
print('working directory:', Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# INPUT CELL: this must report a CUDA GPU before continuing.
import torch
assert torch.cuda.is_available(), 'Enable Runtime > Change runtime type > GPU before running the full experiment.'
print(torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory // 2**30, 'GiB')

In [ ]:
# INPUT CELL: create a Hugging Face token with accepted LLaMA/Gemma licenses, then add it to Colab Secrets as HF_TOKEN.
from google.colab import userdata
token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN to Colab Secrets. The value is not printed or written to disk.'
os.environ['HF_TOKEN'] = token
from huggingface_hub import HfApi
api = HfApi(token=token)
for model_id in ['meta-llama/Llama-3.2-1B-Instruct', 'google/gemma-3-1b-it', 'google/mt5-small']:
    info = api.model_info(model_id)
    print(model_id, info.sha)

In [ ]:
# Persist all non-weight artifacts across runtime interruptions.
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/mt_translation_assignment/results')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(RUN_DIR)

In [ ]:
# Prepare the fixed full dataset. Re-running is deterministic and refreshes only matching manifests.
import subprocess
subprocess.run(['python', 'scripts/run_pipeline.py', '--stage', 'prepare', '--output-dir', str(RUN_DIR)], check=True)

In [ ]:
# Train the custom Transformer and fine-tune mT5. Checkpoints are persisted and reused when fingerprints match.
subprocess.run(['python', 'scripts/run_pipeline.py', '--stage', 'train', '--output-dir', str(RUN_DIR)], check=True)

In [ ]:
# Generate identical ordered test-set predictions. LLaMA and Gemma are loaded sequentially.
subprocess.run(['python', 'scripts/run_pipeline.py', '--stage', 'generate', '--output-dir', str(RUN_DIR)], check=True)

In [ ]:
# Evaluate, create plots, create the bilingual review queue, and generate reports from actual saved artifacts.
subprocess.run(['python', 'scripts/run_pipeline.py', '--stage', 'evaluate', '--output-dir', str(RUN_DIR), '--reports-dir', str(RUN_DIR / 'reports')], check=True)
from IPython.display import display
import pandas as pd
display(pd.read_csv(RUN_DIR / 'metrics.csv'))
display(list((RUN_DIR / 'plots').glob('*')))

In [ ]:
# Separate enabled-sampling experiment: temperatures 0.3, 0.7, 1.0 and seeds 42/43/44.
# It is intentionally not merged into deterministic benchmark metrics.
subprocess.run(['python', 'scripts/run_pipeline.py', '--stage', 'temperature', '--output-dir', str(RUN_DIR), '--reports-dir', str(RUN_DIR / 'reports')], check=True)
display(pd.read_csv(RUN_DIR / 'temperature' / 'aggregated_metrics.csv'))

In [ ]:
# Review actual examples. Add bilingual human labels/notes to ethical_review.jsonl; challenge cases stay outside scores.
review = RUN_DIR / 'ethical_review.jsonl'
print(review)
print(review.read_text(encoding='utf-8')[:4000])

In [ ]:
# Export reports/results without credentials, caches, or model weights.
import zipfile
zip_path = Path('/content/drive/MyDrive/mt_translation_assignment/mt_translation_results.zip')
excluded = {'.cache', 'models'}
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob('*'):
        if path.is_file() and not any(part in excluded for part in path.parts) and path.suffix not in {'.pt', '.bin', '.safetensors'}:
            archive.write(path, path.relative_to(RUN_DIR))
print('download from:', zip_path)